# Inspect individual handwriting-aware predictions

Choose a method, WD/WI, fold, and validation sample below. This notebook loads an existing recognizer checkpoint, runs **one real ED recording**, and shows how its predicted token IDs become text. It supports handwriting-aware **Bigram, BPE and Unigram**. No training is performed.

The dataset must be extracted under `data/tva/ed/gold_wd` and `gold_wi`; the chosen saved run and checkpoint must also exist. The default uses CPU and the best-CER checkpoint.


In [1]:
from pathlib import Path
import json

print('Importing PyTorch and TVA...', flush=True)
import torch
import yaml
import jiwer
from IPython.display import Markdown, display
from tva.model import BaseModel
from tva.dataset import HRDataset
from tva.tokenizer_factory import get_tokenizer, resolve_tokenizer_path
from tva.decoder_ctc import BestPath

ROOT = Path.cwd()  # Run from the TVA repository folder.
METHOD = 'unigram'  # 'bigram', 'bpe' or 'unigram'
SPLIT = 'wi'       # 'wd' or 'wi'
FOLD = 0           # Choose a fold with an existing saved run.
SAMPLE_INDEX = 1   # Index in this fold's validation set.
CHECKPOINT = 'best_cer.pth'  # Also: 'best_wer.pth' or 'latest.pth'
DEVICE = 'cuda'
FRAME_COUNT = 30   # Number of individual prediction frames to display.

run_names = {'bigram': 'handwriting', 'bpe': 'handwriting_bpe', 'unigram': 'handwriting_unigram'}
config_names = {'bigram': 'bigram_handwriting', 'bpe': 'bpe_handwriting', 'unigram': 'unigram_handwriting'}
print('Ready. Change the settings above, then run the remaining cells.')

Importing PyTorch and TVA...
Ready. Change the settings above, then run the remaining cells.


## Load the matching tokenizer, model and sample

The saved run supplies the model's settings. The current method config locates the same frozen tokenizer after the artifact-directory reorganization; its tokenizer key and alphabet must match the saved run. Validation uses TVA's standard normalization/padding, with augmentation disabled.


In [2]:
run_dir = ROOT / f'results/thesis/ed/{run_names[METHOD]}_{SPLIT}/{FOLD}'
config_paths = list(run_dir.glob('train_20*.yaml'))
assert len(config_paths) == 1, f'Expected one saved run config in {run_dir}'
config_path = config_paths[0]
saved_config = yaml.safe_load(config_path.read_text())
current_config = yaml.safe_load((ROOT / f'configs/thesis/ed/{config_names[METHOD]}_{SPLIT}.yaml').read_text())
assert saved_config['tokenizer'] == current_config['tokenizer']
assert saved_config['categories'] == current_config['categories']
assert saved_config.get('ed_distribution', SPLIT) == SPLIT and saved_config['idx_fold'] == FOLD

tokenizer = get_tokenizer(saved_config['tokenizer'])
tokenizer.load(resolve_tokenizer_path(str(ROOT / current_config['dir_tokenizer']), FOLD))
checkpoint_path = run_dir / 'checkpoints' / CHECKPOINT
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
epoch = checkpoint['epoch']
model = BaseModel(saved_config['arch_en'], saved_config['arch_de'], saved_config['num_channel'], tokenizer.size, saved_config['len_seq'])
model.load_state_dict(checkpoint['model'])
model.to(DEVICE).eval()
del checkpoint

dataset = HRDataset(
    str(ROOT / f'data/tva/ed/gold_{SPLIT}/val.json'),
    tokenizer, model.ratio_ds, FOLD,
    len_seq=saved_config['len_seq'], aug=False, cache=False,
)
signal, target_ids = dataset[SAMPLE_INDEX]
annotation = dataset.annos[SAMPLE_INDEX]
reference = annotation['label']
print('Run:', run_dir)
print('Checkpoint:', CHECKPOINT, '| epoch:', epoch)
print('Sample index:', SAMPLE_INDEX, '| record ID:', annotation['id'], '| writer:', annotation['id_writer'])
print('Signal shape:', tuple(signal.shape), '| vocabulary classes:', tokenizer.size)
print('Ground truth:', repr(reference))


2026-10-22 21:48:19.239 | INFO     | tva.handwriting_tokenizers:load:489 - HandwritingUnigramTokenizer is loaded from /home/artellisys/TVA/artifacts/tokenizers/ed/handwriting/unigram/0.json.


Run: /home/artellisys/TVA/results/thesis/ed/handwriting_unigram_wi/0
Checkpoint: best_cer.pth | epoch: 277
Sample index: 1 | record ID: 501 | writer: 6221
Signal shape: (2317, 7) | vocabulary classes: 224
Ground truth: 'But the real question is: how do we evaluate them?'


## Decode the model prediction

The actual model produces a score for each token at each output frame. TVA's `BestPath` selects the highest-scoring ID per frame, collapses adjacent repeats, and decodes token pieces. Blank ID 0 maps to the empty string. Blank-separated repeated tokens survive.

**Predicted pieces below come directly from the model's CTC path.** They are not obtained by re-encoding its decoded text, which could choose a different segmentation. CER/WER here describe only this one recording.


In [3]:
x = signal.T.unsqueeze(0).to(DEVICE)  # [1, channels, time]
with torch.inference_mode():
    scores = model(x)[0, :len(signal) // model.ratio_ds].cpu()

frame_ids = scores.argmax(dim=-1).tolist()
collapsed_ids = torch.unique_consecutive(torch.tensor(frame_ids)).tolist()
predicted_ids = [i for i in collapsed_ids if i != 0]
predicted_pieces = [tokenizer.idx_token[i] for i in predicted_ids]
prediction = BestPath(tokenizer).decode(scores)
assert tokenizer.decode(predicted_ids) == prediction
assert tokenizer.decode(target_ids.tolist()) == reference

print('Ground truth:     ', repr(reference))
print('Prediction:       ', repr(prediction))
print('Target token IDs: ', target_ids.tolist())
print('Target pieces:    ', [tokenizer.idx_token[i] for i in target_ids.tolist()])
print('Predicted IDs:    ', predicted_ids)
print('Predicted pieces: ', predicted_pieces)
print(f'Sample CER: {100 * jiwer.cer(reference, prediction):.2f}%')
print(f'Sample WER: {100 * jiwer.wer(reference, prediction):.2f}%')


Ground truth:      'But the real question is: how do we evaluate them?'
Prediction:        'But the real question is: how dowe evalulee than?'
Target token IDs:  [28, 220, 1, 206, 1, 193, 94, 1, 69, 73, 138, 72, 61, 184, 1, 61, 71, 23, 1, 162, 75, 1, 56, 67, 1, 222, 1, 140, 94, 73, 100, 57, 1, 206, 65, 26]
Target pieces:     ['B', 'ut', ' ', 'the', ' ', 're', 'al', ' ', 'q', 'u', 'es', 't', 'i', 'on', ' ', 'i', 's', ':', ' ', 'ho', 'w', ' ', 'd', 'o', ' ', 'we', ' ', 'ev', 'al', 'u', 'at', 'e', ' ', 'the', 'm', '?']
Predicted IDs:     [28, 220, 1, 206, 1, 193, 94, 1, 69, 73, 138, 72, 61, 184, 1, 61, 71, 23, 1, 162, 75, 1, 56, 67, 222, 1, 140, 94, 73, 171, 57, 1, 205, 96, 26]
Predicted pieces:  ['B', 'ut', ' ', 'the', ' ', 're', 'al', ' ', 'q', 'u', 'es', 't', 'i', 'on', ' ', 'i', 's', ':', ' ', 'ho', 'w', ' ', 'd', 'o', 'we', ' ', 'ev', 'al', 'u', 'le', 'e', ' ', 'th', 'an', '?']
Sample CER: 10.00%
Sample WER: 40.00%


## Inspect individual prediction frames

This table shows the first `FRAME_COUNT` output frames. A frame contributes a token only when its winning ID differs from the previous frame and is not blank. The final prediction above uses **all valid frames**, not only the displayed portion.


In [4]:
lines = [
    '| Frame | Winning ID | Token | Contributes after CTC collapse? |',
    '| ---: | ---: | --- | --- |',
]
for frame, token_id in enumerate(frame_ids[:FRAME_COUNT]):
    piece = '<blank>' if token_id == 0 else repr(tokenizer.idx_token[token_id])
    piece = piece.replace('|', chr(92) + '|')
    kept = token_id != 0 and (frame == 0 or token_id != frame_ids[frame - 1])
    lines.append(f'| {frame} | {token_id} | {piece} | {kept} |')
display(Markdown(chr(10).join(lines)))
print(f'Displaying {min(FRAME_COUNT, len(frame_ids))} of {len(frame_ids)} output frames.')


| Frame | Winning ID | Token | Contributes after CTC collapse? |
| ---: | ---: | --- | --- |
| 0 | 28 | 'B' | True |
| 1 | 0 | <blank> | False |
| 2 | 0 | <blank> | False |
| 3 | 0 | <blank> | False |
| 4 | 0 | <blank> | False |
| 5 | 0 | <blank> | False |
| 6 | 0 | <blank> | False |
| 7 | 0 | <blank> | False |
| 8 | 0 | <blank> | False |
| 9 | 0 | <blank> | False |
| 10 | 0 | <blank> | False |
| 11 | 0 | <blank> | False |
| 12 | 0 | <blank> | False |
| 13 | 0 | <blank> | False |
| 14 | 0 | <blank> | False |
| 15 | 0 | <blank> | False |
| 16 | 0 | <blank> | False |
| 17 | 220 | 'ut' | True |
| 18 | 0 | <blank> | False |
| 19 | 1 | ' ' | True |
| 20 | 1 | ' ' | False |
| 21 | 1 | ' ' | False |
| 22 | 0 | <blank> | False |
| 23 | 0 | <blank> | False |
| 24 | 0 | <blank> | False |
| 25 | 206 | 'the' | True |
| 26 | 0 | <blank> | False |
| 27 | 0 | <blank> | False |
| 28 | 0 | <blank> | False |
| 29 | 0 | <blank> | False |

Displaying 30 of 289 output frames.


## Compare with the prediction saved during training

The original prediction file contains text strings, not frame scores or token IDs. The line below shows its prediction for the same checkpoint epoch and validation index. Fresh single-record inference can differ from the saved batched/mixed-precision result because padding and numerical precision can affect decoding. This notebook does not replace the validated experiment scores.


In [5]:
predictions_path = config_path.with_name(config_path.stem.replace('train_', 'train_preds_', 1) + '.json')
saved_predictions = json.loads(predictions_path.read_text())
assert saved_predictions['labels'] == [record['label'] for record in dataset.annos]
saved_prediction = saved_predictions[str(epoch)][SAMPLE_INDEX]
print('Ground truth:             ', repr(reference))
print('Saved training prediction:', repr(saved_prediction))
print('Fresh single-sample text: ', repr(prediction))
print('Same text:', saved_prediction == prediction)
del saved_predictions


Ground truth:              'But the real question is: how do we evaluate them?'
Saved training prediction: 'But the real question is: how do we evaluate them?'
Fresh single-sample text:  'But the real question is: how dowe evalulee than?'
Same text: False
